# LeetCode #2 — Add Two Numbers: nhập môn Linked List qua phép cộng có nhớ

**Mức độ:** Medium · **Chủ đề:** Linked List, Math, Simulation

Skeleton trên LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
```

Mục tiêu của notebook là giúp bạn hiểu **cách duyệt và xây dựng Linked List (danh sách liên kết)**, đồng thời mô phỏng lại **phép cộng có nhớ (carry)** mà bạn đã học từ tiểu học.

---

## Sau notebook này, bạn sẽ nắm được

1. Linked List là gì, `ListNode` gồm những gì, duyệt nó bằng `while` ra sao.
2. Vì sao chữ số được lưu **ngược** lại giúp bài toán dễ hơn.
3. Cách tách chữ số và số nhớ bằng `% 10` và `// 10` (hoặc `divmod`).
4. Kỹ thuật **dummy node** để xây danh sách kết quả mà không cần xử lý riêng nút đầu.
5. Cách xử lý hai danh sách dài khác nhau và **số nhớ còn sót ở cuối**.
6. Mẹo nhận diện bài "mô phỏng phép tính trên Linked List".
7. Bài tập vận dụng từ dễ đến nâng cao.

# 1. Bài toán Add Two Numbers là gì?

Cho hai Linked List **không rỗng** `l1` và `l2`, mỗi danh sách biểu diễn một **số nguyên không âm**.

- Các chữ số được lưu theo **thứ tự ngược**: nút đầu tiên là **hàng đơn vị**.
- Mỗi nút chứa đúng **một chữ số** (0 → 9).
- Hãy cộng hai số và trả về tổng, cũng dưới dạng Linked List theo thứ tự ngược.

Đề đảm bảo: hai số không có chữ số 0 ở đầu, trừ khi chính số đó là `0`.

**Constraints (theo đề):**

- Số nút trong mỗi danh sách nằm trong khoảng `[1, 100]`.
- `0 <= Node.val <= 9`.

---

## Ví dụ 1

```text
l1 = 2 -> 4 -> 3        nghĩa là số 342
l2 = 5 -> 6 -> 4        nghĩa là số 465
```

`342 + 465 = 807`, viết ngược lại là `7 -> 0 -> 8`.

Kết quả: `[7, 0, 8]`

## Ví dụ 2

```text
l1 = [0], l2 = [0]  ->  [0]
```

## Ví dụ 3

```text
l1 = [9,9,9,9,9,9,9]   (9999999)
l2 = [9,9,9,9]         (9999)
-> [8,9,9,9,0,0,0,1]   (10009998)
```

Kiểm tra nhanh ví dụ 1 bằng số nguyên bình thường. Lưu ý cách "đọc ngược" một list Python: `[2, 4, 3]` đọc từ phải sang trái là `342`.

In [1]:
digits_1 = [2, 4, 3]
digits_2 = [5, 6, 4]

# Đọc ngược list rồi ghép thành chuỗi số
num_1 = int("".join(str(d) for d in reversed(digits_1)))
num_2 = int("".join(str(d) for d in reversed(digits_2)))

print("num_1 =", num_1)
print("num_2 =", num_2)
print("tổng  =", num_1 + num_2)

num_1 = 342
num_2 = 465
tổng  = 807


# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)
2. Tính toán theo từng chữ số (cộng / nhân có nhớ)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [2]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [3]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [4]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [5]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [6]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Tính toán theo từng chữ số (cộng / nhân có nhớ)

### Cộng tay có nhớ

Ta cộng từ **hàng đơn vị** sang trái; mỗi cột: `tổng = a + b + nhớ`, ghi `tổng % 10`, nhớ `tổng // 10` sang cột kế.

```text
      1 1         ← nhớ
      4 8 7
  +   3 6 5
  ---------
      8 5 2
```

In [7]:
def cong_chuoi(x, y):
    i, j, nho, out = len(x) - 1, len(y) - 1, 0, []
    while i >= 0 or j >= 0 or nho:
        a_ = int(x[i]) if i >= 0 else 0
        b_ = int(y[j]) if j >= 0 else 0
        tong = a_ + b_ + nho
        out.append(str(tong % 10))
        nho = tong // 10
        print(f"   cột: {a_} + {b_} + nhớ → {tong}: ghi {tong % 10}, nhớ {nho}")
        i -= 1; j -= 1
    return "".join(reversed(out))

print("487 + 365 =", cong_chuoi("487", "365"))
print("999 + 1 =", cong_chuoi("999", "1"), "(nhớ ở cột cuối tạo thêm chữ số)")

   cột: 7 + 5 + nhớ → 12: ghi 2, nhớ 1
   cột: 8 + 6 + nhớ → 15: ghi 5, nhớ 1
   cột: 4 + 3 + nhớ → 8: ghi 8, nhớ 0
487 + 365 = 852
   cột: 9 + 1 + nhớ → 10: ghi 0, nhớ 1
   cột: 9 + 0 + nhớ → 10: ghi 0, nhớ 1
   cột: 9 + 0 + nhớ → 10: ghi 0, nhớ 1
   cột: 0 + 0 + nhớ → 1: ghi 1, nhớ 0
999 + 1 = 1000 (nhớ ở cột cuối tạo thêm chữ số)


### Nhân tay: chữ số hàng `i` × chữ số hàng `j` rơi vào hàng `i + j`

Đánh số từ **phải** (hàng đơn vị là 0): tích hai chữ số ở hàng `i` và `j` đóng góp vào hàng `i + j` của kết quả. Cộng dồn mọi đóng góp rồi xử lý nhớ một lượt. Tích của số `m` chữ số và `n` chữ số có tối đa `m + n` chữ số.

```text
        1 2 3          hàng:  2 1 0
      ×   4 5                   1 0
      -------
        6 1 5          123 × 5
      4 9 2            123 × 4, dịch một hàng
      -------
      5 5 3 5
```

In [8]:
def nhan_chuoi(x, y):
    cot = [0] * (len(x) + len(y))           # cot[k] = tổng đóng góp vào hàng k (từ phải)
    for i, dx in enumerate(reversed(x)):
        for j, dy in enumerate(reversed(y)):
            cot[i + j] += int(dx) * int(dy)
    nho = 0
    for k in range(len(cot)):               # xử lý nhớ
        nho, cot[k] = divmod(cot[k] + nho, 10)
    s = "".join(map(str, reversed(cot))).lstrip("0")
    return s or "0"

print("123 × 45 =", nhan_chuoi("123", "45"), "| kiểm tra:", 123 * 45)
print("0 × 999 =", nhan_chuoi("0", "999"))

123 × 45 = 5535 | kiểm tra: 5535
0 × 999 = 0


Lỗi hay gặp: quên **nhớ ở cột cuối** (`999 + 1`); đọc chữ số từ trái trong khi phải cộng từ phải; quên bỏ số 0 đầu (`"00"` → `"0"`); hai số dài khác nhau — hết chữ số thì coi là 0.

## 2.3 Áp dụng vào bài này

Hai số được lưu dưới dạng **danh sách liên kết**, chữ số hàng đơn vị nằm ở **đầu** danh sách — đúng thứ tự ta cộng tay (từ phải sang trái). Vì vậy chỉ cần **duyệt đồng thời** hai danh sách, mỗi bước cộng hai chữ số cùng **biến nhớ**, tạo một nút mới chứa `tổng % 10`. Danh sách kết quả được dựng bằng **nút giả** (dummy) để khỏi xử lý riêng nút đầu, và đừng quên phần **nhớ ở cột cuối** (`5 + 5 = 10` cần thêm nút `1`).

# 3. Nhìn lại phép cộng bằng tay

Hãy cộng `342 + 465` theo cột, như hồi tiểu học. Ta **bắt đầu từ hàng đơn vị** (bên phải):

| Hàng | Chữ số của 342 | Chữ số của 465 | Nhớ vào | Tổng cột | Viết xuống | Nhớ ra |
|---|---|---|---|---|---|---|
| đơn vị | 2 | 5 | 0 | 7 | 7 | 0 |
| chục | 4 | 6 | 0 | 10 | 0 | 1 |
| trăm | 3 | 4 | 1 | 8 | 8 | 0 |

Đọc cột "Viết xuống" từ trên xuống: `7, 0, 8` — **đúng y hệt** thứ tự đầu ra của đề!

> Đây là lý do đề lưu chữ số ngược: nút đầu tiên chính là hàng đơn vị, nên **duyệt từ đầu danh sách = cộng từ phải sang trái**. Ta không cần đảo gì cả.

## Tách "chữ số viết xuống" và "số nhớ"

Khi tổng một cột là `total`:

- Chữ số viết xuống là `total % 10` (phần dư khi chia 10).
- Số nhớ sang cột kế tiếp là `total // 10` (phần nguyên khi chia 10).

Ví dụ `total = 13`: viết `3`, nhớ `1`.

Lưu ý: `//` là **chia lấy phần nguyên**, còn `/` cho ra số thực (`13 / 10 = 1.3`) — dùng nhầm sẽ sai.

In [9]:
total = 13
print("total % 10  =", total % 10)    # chữ số viết xuống
print("total // 10 =", total // 10)   # số nhớ
print("total / 10  =", total / 10)    # số thực, KHÔNG dùng

# divmod trả về cả hai cùng lúc: (phần nguyên, phần dư)
carry, digit = divmod(total, 10)
print("divmod(13, 10) -> carry =", carry, ", digit =", digit)

total % 10  =

 3
total // 10 = 1
total / 10  = 1.3
divmod(13, 10) -> carry = 1 , digit = 3


Với hai chữ số (mỗi số ≤ 9) và số nhớ (≤ 1), tổng một cột lớn nhất là `9 + 9 + 1 = 19`. Vì vậy **số nhớ luôn chỉ là 0 hoặc 1**.

In [10]:
for total in [0, 7, 9, 10, 18, 19]:
    carry, digit = divmod(total, 10)
    print(f"total = {total:2d} -> viết {digit}, nhớ {carry}")

total =  0 -> viết 0, nhớ 0
total =  7 -> viết 7, nhớ 0
total =  9 -> viết 9, nhớ 0
total = 10 -> viết 0, nhớ 1
total = 18 -> viết 8, nhớ 1
total = 19 -> viết 9, nhớ 1


# 4. Linked List là gì?

**Linked List (danh sách liên kết)** là một chuỗi các **nút (node)**. Mỗi nút giữ:

- `val`: giá trị (ở bài này là một chữ số).
- `next`: "mũi tên" chỉ sang nút kế tiếp. Nút cuối có `next = None`.

```text
 [2] -> [4] -> [3] -> None
```

Khác với list Python (`[2, 4, 3]`), ta **không truy cập được** `l1[2]` ngay lập tức. Muốn tới nút thứ 3, phải đi lần lượt qua `next`.

Trên LeetCode, class `ListNode` được định nghĩa sẵn. Ở môi trường Python bình thường, ta tự viết:

In [11]:
from typing import Optional, List


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    def __repr__(self):
        return f"ListNode({self.val})"


# Tạo 3 nút rời rạc
a = ListNode(2)
b = ListNode(4)
c = ListNode(3)

# Nối chúng lại: a -> b -> c
a.next = b
b.next = c

print(a, "->", a.next, "->", a.next.next, "->", a.next.next.next)

ListNode(2) -> ListNode(4) -> ListNode(3) -> None


## Duyệt Linked List bằng `while`

Mẫu duyệt chuẩn:

```python
node = head
while node:          # còn nút thì còn đi
    ... node.val ...
    node = node.next # bước sang nút kế tiếp
```

`while node` dừng khi `node` trở thành `None` — tức là đã đi qua nút cuối.

Đừng quên dòng `node = node.next`; thiếu nó vòng lặp sẽ chạy mãi mãi.

In [12]:
node = a
step = 0
while node:
    print(f"bước {step}: node.val = {node.val}")
    node = node.next
    step += 1
print("node bây giờ là:", node)

bước 0: node.val = 2
bước 1: node.val = 4
bước 2: node.val = 3
node bây giờ là: None


## Hai hàm tiện ích: list Python ↔ Linked List

Để test cho nhanh, ta viết hai hàm chuyển đổi. Chúng **không phải** một phần lời giải, chỉ giúp nhập/xuất dữ liệu.

`build_list` dùng một nút giả (`dummy`) để nối dần — kỹ thuật này sẽ được giải thích kỹ ở mục 7.

In [13]:
def build_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode]) -> List[int]:
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


l1 = build_list([2, 4, 3])
print(to_list(l1))
print(to_list(build_list([])))  # danh sách rỗng -> None -> []

[2, 4, 3]
[]


# 5. Cách dễ nghĩ nhất: đổi sang số nguyên rồi cộng

Ý tưởng trực giác:

1. Đọc `l1` thành số nguyên `n1`, đọc `l2` thành `n2`.
2. Tính `n1 + n2`.
3. Tách tổng thành từng chữ số (từ hàng đơn vị) và tạo Linked List mới.

Đọc số từ danh sách ngược: chữ số ở vị trí `k` có giá trị `digit * 10**k`.

In [14]:
def list_to_int(head: Optional[ListNode]) -> int:
    number = 0
    place = 1            # 1, 10, 100, ...
    while head:
        number += head.val * place
        place *= 10
        head = head.next
    return number


def int_to_list(number: int) -> Optional[ListNode]:
    if number == 0:
        return ListNode(0)
    dummy = ListNode()
    tail = dummy
    while number > 0:
        tail.next = ListNode(number % 10)
        tail = tail.next
        number //= 10
    return dummy.next


def add_two_numbers_via_int(l1, l2):
    return int_to_list(list_to_int(l1) + list_to_int(l2))


print(list_to_int(build_list([2, 4, 3])))
print(to_list(add_two_numbers_via_int(build_list([2, 4, 3]), build_list([5, 6, 4]))))
print(to_list(add_two_numbers_via_int(build_list([0]), build_list([0]))))
print(to_list(add_two_numbers_via_int(build_list([9] * 7), build_list([9] * 4))))

342
[7, 0, 8]
[0]
[8, 9, 9, 9, 0, 0, 0, 1]


## Vấn đề của cách này

Trong Python, cách trên **chạy được** vì `int` của Python lớn tùy ý.

Nhưng:

- Đề cho tới **100 chữ số**. Trong Java/C++, `long` chỉ chứa khoảng 19 chữ số → **tràn số (overflow)**.
- Phỏng vấn viên thường muốn thấy bạn **thao tác trên Linked List**, không phải né nó.
- Ta đi qua dữ liệu tới 3 lần (đọc `l1`, đọc `l2`, tạo kết quả).

Hãy nhìn lại bảng cộng tay ở mục 3: mỗi cột chỉ cần **2 chữ số + 1 số nhớ**. Ta không bao giờ cần cả con số lớn!

# 6. Ý tưởng chính: cộng từng cột trong lúc duyệt

Ta đi song song trên `l1` và `l2`, mỗi bước xử lý **một cột**:

```text
x = chữ số hiện tại của l1 (hoặc 0 nếu l1 đã hết)
y = chữ số hiện tại của l2 (hoặc 0 nếu l2 đã hết)
total = x + y + carry
digit = total % 10      -> tạo nút mới
carry = total // 10     -> mang sang cột sau
```

Ba câu hỏi quan trọng:

1. **Hai danh sách dài khác nhau thì sao?** Coi phần thiếu là `0`. Ví dụ `999 + 99` giống `999 + 099`.
2. **Khi nào dừng?** Khi **cả hai** danh sách đã hết **và** không còn số nhớ.
3. **Số nhớ ở cột cuối?** `99 + 1 = 100`: sau khi hết chữ số, vẫn còn nhớ `1` → phải tạo thêm nút `1`.

Vì vậy điều kiện vòng lặp là:

```python
while l1 or l2 or carry:
```

In [15]:
# Mô phỏng câu hỏi 1: lấy chữ số an toàn khi nút có thể là None
node = None
x = node.val if node else 0
print("x =", x)

node = ListNode(7)
x = node.val if node else 0
print("x =", x)

x = 0
x = 7


In [16]:
# Mô phỏng câu hỏi 3: điều kiện dừng
for l1_done, l2_done, carry in [(False, True, 0), (True, True, 1), (True, True, 0)]:
    l1_tmp = None if l1_done else "còn"
    l2_tmp = None if l2_done else "còn"
    keep_going = bool(l1_tmp or l2_tmp or carry)
    print(f"l1 hết={l1_done}, l2 hết={l2_done}, carry={carry} -> tiếp tục? {keep_going}")

l1 hết=False, l2 hết=True, carry=0 -> tiếp tục? True
l1 hết=True, l2 hết=True, carry=1 -> tiếp tục? True
l1 hết=True, l2 hết=True, carry=0 -> tiếp tục? False


# 7. Kỹ thuật Dummy Node

Ta cần **xây dựng** một Linked List mới, nối từng nút vào cuối.

Vấn đề: lúc đầu danh sách rỗng, chưa có nút nào để nối vào. Nếu không có mẹo, ta phải viết:

```python
if head is None:
    head = new_node
    tail = new_node
else:
    tail.next = new_node
    tail = new_node
```

Mẹo **dummy node**: tạo trước một nút giả. Khi đó luôn có chỗ để nối, và mọi bước đều giống nhau:

```python
dummy = ListNode()   # nút giả, giá trị không quan trọng
tail = dummy         # tail luôn trỏ vào nút cuối hiện tại
...
tail.next = ListNode(digit)
tail = tail.next
...
return dummy.next    # bỏ nút giả, trả về nút thật đầu tiên
```

In [17]:
dummy = ListNode(-1)
tail = dummy

for digit in [7, 0, 8]:
    tail.next = ListNode(digit)
    tail = tail.next
    print("sau khi thêm", digit, "-> toàn bộ (kể cả dummy):", to_list(dummy))

print("Trả về dummy.next:", to_list(dummy.next))

sau khi thêm

 7 -> toàn bộ (kể cả dummy): [-1, 7]
sau khi thêm 0 -> toàn bộ (kể cả dummy): [-1, 7, 0]
sau khi thêm 8 -> toàn bộ (kể cả dummy): [-1, 7, 0, 8]
Trả về dummy.next: [7, 0, 8]


# 8. Lời giải chính

Ghép mục 6 và mục 7 lại:

1. Tạo `dummy`, `tail = dummy`, `carry = 0`.
2. Lặp khi `l1 or l2 or carry`:
   - Lấy `x`, `y` (0 nếu hết).
   - `carry, digit = divmod(x + y + carry, 10)`.
   - Nối nút `digit` vào `tail`, dời `tail`.
   - Dời `l1`, `l2` sang nút kế (nếu còn).
3. Trả về `dummy.next`.

In [18]:
def add_two_numbers(l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    carry = 0

    while l1 or l2 or carry:
        x = l1.val if l1 else 0
        y = l2.val if l2 else 0

        carry, digit = divmod(x + y + carry, 10)

        tail.next = ListNode(digit)
        tail = tail.next

        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None

    return dummy.next


print(to_list(add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))))
print(to_list(add_two_numbers(build_list([0]), build_list([0]))))
print(to_list(add_two_numbers(build_list([9, 9, 9, 9, 9, 9, 9]), build_list([9, 9, 9, 9]))))

[7, 0, 8]
[0]
[8, 9, 9, 9, 0, 0, 0, 1]


# 9. Mô phỏng từng bước

Xét `l1 = [9, 9, 9]` (số 999) và `l2 = [9, 9]` (số 99). Tổng là `1098` → kết quả `[8, 9, 0, 1]`.

Ví dụ này đủ "khó" vì có cả **hai danh sách dài khác nhau** và **số nhớ ở cuối**.

| Bước | x | y | carry vào | total | digit | carry ra | Kết quả đến lúc này |
|---|---|---|---|---|---|---|---|
| 0 | 9 | 9 | 0 | 18 | 8 | 1 | 8 |
| 1 | 9 | 9 | 1 | 19 | 9 | 1 | 8 → 9 |
| 2 | 9 | 0 (l2 hết) | 1 | 10 | 0 | 1 | 8 → 9 → 0 |
| 3 | 0 (l1 hết) | 0 | 1 | 1 | 1 | 0 | 8 → 9 → 0 → 1 |

Sau bước 3: `l1`, `l2` đều `None`, `carry = 0` → dừng.

Nếu điều kiện vòng lặp chỉ là `while l1 or l2`, ta sẽ **mất bước 3** và trả về `[8, 9, 0]` — sai!

In [19]:
def trace_add_two_numbers(l1, l2):
    dummy = ListNode()
    tail = dummy
    carry = 0
    step = 0

    while l1 or l2 or carry:
        x = l1.val if l1 else 0
        y = l2.val if l2 else 0
        carry_in = carry
        carry, digit = divmod(x + y + carry_in, 10)

        tail.next = ListNode(digit)
        tail = tail.next

        print("=" * 50)
        print(f"Bước {step}")
        print(f"  x = {x}{'  (l1 đã hết)' if not l1 else ''}")
        print(f"  y = {y}{'  (l2 đã hết)' if not l2 else ''}")
        print(f"  total = {x} + {y} + {carry_in} = {x + y + carry_in}")
        print(f"  viết {digit}, nhớ {carry}")
        print(f"  kết quả hiện tại: {to_list(dummy.next)}")

        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
        step += 1

    print("=" * 50)
    print("Dừng vì l1, l2 đều hết và carry = 0")
    return dummy.next


result = trace_add_two_numbers(build_list([9, 9, 9]), build_list([9, 9]))
print("Đáp án:", to_list(result))


Bước 0
  x = 9
  y = 9
  total = 9 + 9 + 0 = 18
  viết 8, nhớ 1
  kết quả hiện tại: [8]
Bước 1
  x = 9
  y = 9
  total = 9 + 9 + 1 = 19
  viết 9, nhớ 1
  kết quả hiện tại: [8, 9]
Bước 2
  x = 9
  y = 0  (l2 đã hết)
  total = 9 + 0 + 1 = 10
  viết 0, nhớ 1
  kết quả hiện tại: [8, 9, 0]
Bước 3
  x = 0  (l1 đã hết)
  y = 0  (l2 đã hết)
  total = 0 + 0 + 1 = 1
  viết 1, nhớ 0
  kết quả hiện tại: [8, 9, 0, 1]
Dừng vì l1, l2 đều hết và carry = 0
Đáp án: [8, 9, 0, 1]


# 10. Phiên bản đúng format LeetCode

Trên LeetCode, `ListNode` đã có sẵn, bạn chỉ dán phần `class Solution`. Ở đây ta giữ nguyên định nghĩa `ListNode` phía trên để chạy thử.

Code cần nộp:

In [20]:
from typing import Optional


class Solution:
    def addTwoNumbers(self, l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode()
        tail = dummy
        carry = 0

        while l1 or l2 or carry:
            x = l1.val if l1 else 0
            y = l2.val if l2 else 0
            carry, digit = divmod(x + y + carry, 10)

            tail.next = ListNode(digit)
            tail = tail.next

            l1 = l1.next if l1 else None
            l2 = l2.next if l2 else None

        return dummy.next

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<32} PASS  {actual}")


sol = Solution()

def run(a, b):
    return to_list(sol.addTwoNumbers(build_list(a), build_list(b)))

check("Ví dụ 1: 342 + 465", run([2, 4, 3], [5, 6, 4]), [7, 0, 8])
check("Ví dụ 2: 0 + 0", run([0], [0]), [0])
check("Ví dụ 3", run([9] * 7, [9] * 4), [8, 9, 9, 9, 0, 0, 0, 1])
check("Dài khác nhau, không nhớ", run([1], [2, 3, 4]), [3, 3, 4])
check("Nhớ ở cuối: 5 + 5", run([5], [5]), [0, 1])
check("Nhớ lan dài: 9999 + 1", run([9, 9, 9, 9], [1]), [0, 0, 0, 0, 1])
check("Một số là 0", run([0], [7, 3]), [7, 3])

Ví dụ 1: 342 + 465               PASS  [7, 0, 8]


Ví dụ 2: 0 + 0                   PASS  [0]
Ví dụ 3                          PASS  [8, 9, 9, 9, 0, 0, 0, 1]
Dài khác nhau, không nhớ         PASS  [3, 3, 4]
Nhớ ở cuối: 5 + 5                PASS  [0, 1]
Nhớ lan dài: 9999 + 1            PASS  [0, 0, 0, 0, 1]
Một số là 0                      PASS  [7, 3]


## Đối chiếu với cách đổi sang số nguyên

Ta sinh ngẫu nhiên nhiều cặp danh sách (đúng constraints: 1–100 chữ số, không có số 0 thừa ở đầu) và so sánh kết quả hai cách.

In [22]:
import random

random.seed(0)

def random_number_digits(max_len=100):
    n = random.randint(1, max_len)
    if n == 1:
        return [random.randint(0, 9)]
    digits = [random.randint(0, 9) for _ in range(n - 1)]
    digits.append(random.randint(1, 9))  # chữ số cao nhất (ở cuối list) khác 0
    return digits

for _ in range(500):
    a, b = random_number_digits(), random_number_digits()
    fast = run(a, b)
    slow = to_list(add_two_numbers_via_int(build_list(a), build_list(b)))
    assert fast == slow, (a, b)

print("500 test ngẫu nhiên: hai cách cho cùng kết quả")

500 test ngẫu nhiên: hai cách cho cùng kết quả


# 11. Vì sao thuật toán đúng?

**Bất biến (invariant):** sau khi xử lý xong `k` cột (k bước lặp):

- Danh sách kết quả chứa đúng `k` chữ số thấp nhất của tổng.
- `carry` là số nhớ đi vào cột thứ `k`.

Đây chính là định nghĩa của phép cộng theo cột, nên mỗi bước ta chỉ làm đúng việc mà người cộng bằng tay làm.

**Không bỏ sót:** vòng lặp chỉ dừng khi cả hai danh sách đã hết **và** `carry = 0` — tức không còn gì để cộng thêm.

**Không có số 0 thừa ở đầu:** mỗi nút được tạo đều ứng với một cột có thật (còn chữ số hoặc còn nhớ). Ngoại lệ duy nhất là `0 + 0`, cho ra đúng `[0]`.

# 12. Độ phức tạp

Gọi `m` là độ dài `l1`, `n` là độ dài `l2`.

- Vòng lặp chạy `max(m, n)` lần, cộng thêm tối đa 1 lần cho số nhớ cuối.
- Mỗi lần lặp làm vài phép tính cố định.

```text
Time complexity:  O(max(m, n))
Space complexity: O(max(m, n)) cho danh sách kết quả
                  O(1) bộ nhớ phụ (ngoài kết quả)
```

| Cách làm | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| Đổi sang số nguyên | O(m + n) | O(1)* | *Python big int; ngôn ngữ khác bị tràn số |
| Cộng từng cột | O(max(m, n)) | O(1) | Cách chuẩn, chạy được mọi ngôn ngữ |

Không thể nhanh hơn `O(max(m, n))` vì ta buộc phải đọc mọi chữ số.

# 13. Những lỗi dễ mắc

## Lỗi 1: Quên số nhớ ở cột cuối

Sai:

```python
while l1 or l2:
    ...
```

Với `5 + 5`, vòng lặp dừng sau cột đầu tiên, mất chữ số `1`.

In [23]:
def wrong_forget_final_carry(l1, l2):
    dummy = ListNode()
    tail = dummy
    carry = 0
    while l1 or l2:                      # thiếu "or carry"
        x = l1.val if l1 else 0
        y = l2.val if l2 else 0
        carry, digit = divmod(x + y + carry, 10)
        tail.next = ListNode(digit)
        tail = tail.next
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    return dummy.next

print("Sai:", to_list(wrong_forget_final_carry(build_list([5]), build_list([5]))))
print("Đúng:", to_list(add_two_numbers(build_list([5]), build_list([5]))))

Sai: [0]
Đúng: [0, 1]


## Lỗi 2: Dừng khi một danh sách hết

Sai:

```python
while l1 and l2:
    ...
```

Với `[1] + [2, 3, 4]`, vòng lặp dừng sau 1 bước vì `l1` hết, bỏ mất `3, 4`.

In [24]:
def wrong_stop_early(l1, l2):
    dummy = ListNode()
    tail = dummy
    carry = 0
    while l1 and l2:                     # dừng quá sớm
        carry, digit = divmod(l1.val + l2.val + carry, 10)
        tail.next = ListNode(digit)
        tail = tail.next
        l1, l2 = l1.next, l2.next
    return dummy.next

print("Sai:", to_list(wrong_stop_early(build_list([1]), build_list([2, 3, 4]))))
print("Đúng:", to_list(add_two_numbers(build_list([1]), build_list([2, 3, 4]))))

Sai: [3]
Đúng: [3, 3, 4]


## Lỗi 3: Gọi `.val` hoặc `.next` trên `None`

Sai:

```python
x = l1.val          # l1 có thể đã là None
l1 = l1.next        # AttributeError: 'NoneType' object has no attribute 'next'
```

Đúng: luôn kiểm tra trước, `x = l1.val if l1 else 0`.

## Lỗi 4: Trả về `dummy` thay vì `dummy.next`

`dummy` là nút giả. Trả về nó sẽ thêm một chữ số rác ở đầu kết quả.

## Lỗi 5: Quên dời con trỏ

Thiếu `tail = tail.next` → mọi nút mới đều ghi đè vào cùng một chỗ, chỉ còn nút cuối cùng.
Thiếu `l1 = l1.next` → vòng lặp không bao giờ dừng.

## Lỗi 6: Dùng `/` thay vì `//`

`total / 10` cho ra số thực (`1.8`), làm hỏng cả `carry` lẫn giá trị nút.

# 14. Mẹo nhận diện pattern

Hãy nghĩ tới **mô phỏng phép cộng theo cột + dummy node** khi thấy:

- Số rất lớn được cho dưới dạng **danh sách chữ số** (Linked List, mảng, chuỗi).
- Đề yêu cầu cộng / trừ / nhân mà **không được** đổi sang số nguyên.
- Đề yêu cầu **trả về một Linked List mới** → nghĩ ngay tới `dummy` + `tail`.

Mẫu tư duy chung:

```text
carry = 0
while còn chữ số ở bên nào đó or carry:
    lấy chữ số (thiếu thì 0)
    carry, digit = divmod(tổng, cơ số)
    ghi digit
return kết quả
```

Các bài cùng họ trên LeetCode: **#67 Add Binary**, **#415 Add Strings**, **#989 Add to Array-Form of Integer**, **#445 Add Two Numbers II** (chữ số theo thứ tự xuôi).

# 15. Biến thể 1: Cộng hai chuỗi số (Add Strings)

Cùng tư duy, nhưng dữ liệu là chuỗi và chữ số theo **thứ tự xuôi** (`"342"`). Ta duyệt từ **cuối chuỗi** về đầu bằng chỉ số `i`, `j`.

In [25]:
def add_strings(a: str, b: str) -> str:
    i, j = len(a) - 1, len(b) - 1
    carry = 0
    digits = []

    while i >= 0 or j >= 0 or carry:
        x = int(a[i]) if i >= 0 else 0
        y = int(b[j]) if j >= 0 else 0
        carry, digit = divmod(x + y + carry, 10)
        digits.append(str(digit))
        i -= 1
        j -= 1

    return "".join(reversed(digits))

print(add_strings("342", "465"))
print(add_strings("999", "99"))
print(add_strings("0", "0"))

807
1098
0


# 16. Biến thể 2: Cơ số bất kỳ

Thay `10` bằng `base`. Với `base = 2`, đây chính là cộng nhị phân.

Ví dụ: `[1, 1]` (nhị phân `11` = 3) cộng `[1]` (= 1) → `[0, 0, 1]` (nhị phân `100` = 4).

In [26]:
def add_lists_base(l1, l2, base=10):
    dummy = ListNode()
    tail = dummy
    carry = 0
    while l1 or l2 or carry:
        x = l1.val if l1 else 0
        y = l2.val if l2 else 0
        carry, digit = divmod(x + y + carry, base)
        tail.next = ListNode(digit)
        tail = tail.next
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None
    return dummy.next

print(to_list(add_lists_base(build_list([1, 1]), build_list([1]), base=2)))
print(to_list(add_lists_base(build_list([2, 4, 3]), build_list([5, 6, 4]), base=10)))

[0, 0, 1]
[7, 0, 8]


# 17. Template tư duy

```python
dummy = ListNode()
tail = dummy
carry = 0

while l1 or l2 or carry:
    x = l1.val if l1 else 0
    y = l2.val if l2 else 0

    carry, digit = divmod(x + y + carry, 10)

    tail.next = ListNode(digit)
    tail = tail.next

    l1 = l1.next if l1 else None
    l2 = l2.next if l2 else None

return dummy.next
```

Ba điều cần nhớ: **thiếu thì coi là 0**, **còn nhớ thì chưa dừng**, **trả về `dummy.next`**.

# 18. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Hãy lập bảng trace (giống mục 9) cho:

```text
l1 = [5, 6]       (số 65)
l2 = [5, 4, 9]    (số 945)
```

Kết quả là gì? Có bao nhiêu bước lặp? Bước nào có `carry ra = 1`?

## Bài 2 — Đoán output

Đoán kết quả trước, sau đó tự tạo cell để kiểm tra:

```python
print(divmod(19, 10))
print(divmod(7, 10))
print(divmod(10, 2))
print(to_list(build_list([4, 0, 0])))
```

## Bài 3 — Đếm độ dài Linked List

Viết hàm trả về số nút trong danh sách.

```python
count_nodes(build_list([2, 4, 3])) -> 3
count_nodes(None) -> 0
```

In [27]:
def exercise_3_count_nodes(head: Optional[ListNode]) -> int:
    # TODO:
    # Tạo biến đếm
    # Duyệt bằng while head
    # Mỗi bước tăng biến đếm và dời head
    pass


# Bỏ comment để test sau khi làm xong
# assert exercise_3_count_nodes(build_list([2, 4, 3])) == 3
# assert exercise_3_count_nodes(None) == 0

## Bài 4 — Điền chỗ trống cho lời giải chính

In [28]:
def exercise_4_add_two_numbers(l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    carry = 0

    # TODO 1: viết điều kiện vòng lặp đúng (nhớ cả carry)
    # TODO 2: lấy x, y an toàn khi l1 / l2 là None
    # TODO 3: tính carry và digit bằng divmod
    # TODO 4: nối nút digit vào tail và dời tail
    # TODO 5: dời l1, l2 sang nút kế tiếp
    pass


# assert to_list(exercise_4_add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))) == [7, 0, 8]
# assert to_list(exercise_4_add_two_numbers(build_list([9, 9]), build_list([1]))) == [0, 0, 1]

## Bài 5 — Biến thể: Add to Array-Form (LeetCode #989)

Cho mảng `num` biểu diễn một số theo **thứ tự xuôi** (`[1, 2, 0, 0]` là `1200`) và số nguyên `k`. Trả về mảng chữ số (thứ tự xuôi) của `num + k`.

```python
add_to_array_form([1, 2, 0, 0], 34)  -> [1, 2, 3, 4]
add_to_array_form([2, 7, 4], 181)    -> [4, 5, 5]
add_to_array_form([9, 9], 1)         -> [1, 0, 0]
```

Yêu cầu: không đổi `num` thành số nguyên.

In [29]:
def exercise_5_add_to_array_form(num: List[int], k: int) -> List[int]:
    # TODO
    pass


# assert exercise_5_add_to_array_form([1, 2, 0, 0], 34) == [1, 2, 3, 4]
# assert exercise_5_add_to_array_form([2, 7, 4], 181) == [4, 5, 5]
# assert exercise_5_add_to_array_form([9, 9], 1) == [1, 0, 0]

## Bài 6 — Mini challenge: Add Two Numbers II (LeetCode #445)

Giống bài gốc, nhưng chữ số được lưu theo **thứ tự xuôi**: nút đầu là chữ số **cao nhất**.

```text
l1 = [7, 2, 4, 3]   (7243)
l2 = [5, 6, 4]      (564)
-> [7, 8, 0, 7]     (7807)
```

Ràng buộc thêm: không được đảo ngược danh sách đầu vào.

In [30]:
def exercise_6_add_two_numbers_ii(l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_6_add_two_numbers_ii(build_list([7, 2, 4, 3]), build_list([5, 6, 4]))) == [7, 8, 0, 7]
# assert to_list(exercise_6_add_two_numbers_ii(build_list([5]), build_list([5]))) == [1, 0]

# 19. Gợi ý

**Bài 1**
- Mức 1: viết `65` và `945` thẳng hàng theo cột, cộng từ phải sang.
- Mức 2: danh sách ngắn hơn hết trước thì chữ số thiếu là `0`.
- Mức 3: đừng quên bước cuối chỉ còn số nhớ; tổng đúng là `1010`.

**Bài 3**
- Mức 1: giống hàm `to_list`, nhưng đếm thay vì lưu.
- Mức 2: `count = 0`, `while head:`.
- Mức 3: `count += 1; head = head.next`.

**Bài 4**
- Mức 1: so lại với template ở mục 17.
- Mức 2: điều kiện là `l1 or l2 or carry`.
- Mức 3: `carry, digit = divmod(x + y + carry, 10)`.

**Bài 5**
- Mức 1: mảng xuôi → duyệt từ cuối mảng (giống `add_strings`).
- Mức 2: có thể coi `k` là "số nhớ" ban đầu! `k % 10` là chữ số thấp nhất của `k`.
- Mức 3: `while i >= 0 or k: if i >= 0: k += num[i]; digits.append(k % 10); k //= 10; i -= 1`, cuối cùng đảo ngược.

**Bài 6**
- Mức 1: chữ số thấp nhất nằm ở cuối, mà Linked List chỉ đi được từ đầu.
- Mức 2: đẩy chữ số của mỗi danh sách vào một **stack** (list Python), rồi `pop()` để lấy từ hàng đơn vị.
- Mức 3: khi tạo kết quả, chèn nút mới **vào đầu**: `node = ListNode(digit, head); head = node`.

# 20. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

| Bước | x | y | carry vào | total | digit | carry ra |
|---|---|---|---|---|---|---|
| 0 | 5 | 5 | 0 | 10 | 0 | 1 |
| 1 | 6 | 4 | 1 | 11 | 1 | 1 |
| 2 | 0 | 9 | 1 | 10 | 0 | 1 |
| 3 | 0 | 0 | 1 | 1 | 1 | 0 |

Kết quả `[0, 1, 0, 1]` (số `1010` = 65 + 945). Có **4** bước lặp, 3 bước đầu đều nhớ 1.

In [31]:
print(to_list(add_two_numbers(build_list([5, 6]), build_list([5, 4, 9]))))

[0, 1, 0, 1]


## Đáp án Bài 2

In [32]:
print(divmod(19, 10))                   # (1, 9)
print(divmod(7, 10))                    # (0, 7)
print(divmod(10, 2))                    # (5, 0)
print(to_list(build_list([4, 0, 0])))   # [4, 0, 0]

(1, 9)
(0, 7)
(5, 0)
[4, 0, 0]


## Đáp án Bài 3

In [33]:
def answer_3_count_nodes(head: Optional[ListNode]) -> int:
    count = 0
    while head:
        count += 1
        head = head.next
    return count

assert answer_3_count_nodes(build_list([2, 4, 3])) == 3
assert answer_3_count_nodes(None) == 0
print("Bài 3 PASS")

Bài 3 PASS

## Đáp án Bài 4

In [34]:
def answer_4_add_two_numbers(l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    carry = 0

    while l1 or l2 or carry:
        x = l1.val if l1 else 0
        y = l2.val if l2 else 0
        carry, digit = divmod(x + y + carry, 10)
        tail.next = ListNode(digit)
        tail = tail.next
        l1 = l1.next if l1 else None
        l2 = l2.next if l2 else None

    return dummy.next

assert to_list(answer_4_add_two_numbers(build_list([2, 4, 3]), build_list([5, 6, 4]))) == [7, 0, 8]
assert to_list(answer_4_add_two_numbers(build_list([9, 9]), build_list([1]))) == [0, 0, 1]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

Mẹo: dùng chính `k` làm "số nhớ". Mỗi bước cộng chữ số của `num` vào `k`, lấy `k % 10` làm chữ số kết quả, rồi `k //= 10`.

In [35]:
def answer_5_add_to_array_form(num: List[int], k: int) -> List[int]:
    digits = []
    i = len(num) - 1
    while i >= 0 or k:
        if i >= 0:
            k += num[i]
        digits.append(k % 10)
        k //= 10
        i -= 1
    return digits[::-1]

assert answer_5_add_to_array_form([1, 2, 0, 0], 34) == [1, 2, 3, 4]
assert answer_5_add_to_array_form([2, 7, 4], 181) == [4, 5, 5]
assert answer_5_add_to_array_form([9, 9], 1) == [1, 0, 0]
print("Bài 5 PASS")

Bài 5 PASS


## Đáp án Bài 6

Dùng hai stack để lấy chữ số từ hàng đơn vị, và **chèn nút mới vào đầu** kết quả để thứ tự đầu ra là xuôi.

In [36]:
def answer_6_add_two_numbers_ii(l1: Optional[ListNode], l2: Optional[ListNode]) -> Optional[ListNode]:
    stack1, stack2 = [], []
    while l1:
        stack1.append(l1.val)
        l1 = l1.next
    while l2:
        stack2.append(l2.val)
        l2 = l2.next

    head = None
    carry = 0
    while stack1 or stack2 or carry:
        x = stack1.pop() if stack1 else 0
        y = stack2.pop() if stack2 else 0
        carry, digit = divmod(x + y + carry, 10)
        head = ListNode(digit, head)   # chèn vào đầu
    return head

assert to_list(answer_6_add_two_numbers_ii(build_list([7, 2, 4, 3]), build_list([5, 6, 4]))) == [7, 8, 0, 7]
assert to_list(answer_6_add_two_numbers_ii(build_list([5]), build_list([5]))) == [1, 0]
assert to_list(answer_6_add_two_numbers_ii(build_list([0]), build_list([0]))) == [0]
print("Bài 6 PASS")

Bài 6 PASS

# 21. Tổng kết

Add Two Numbers là bài nhập môn rất tốt cho **Linked List**: vừa phải **duyệt** hai danh sách, vừa phải **xây** một danh sách mới.

Cốt lõi:

```python
carry, digit = divmod(x + y + carry, 10)
```

kết hợp với vòng lặp:

```python
while l1 or l2 or carry:
```

---

## Checklist trước khi nộp

- [ ] Điều kiện vòng lặp có `or carry`?
- [ ] Lấy `x`, `y` an toàn khi danh sách đã hết (`if l1 else 0`)?
- [ ] Dùng `//` hoặc `divmod`, không dùng `/`?
- [ ] Có dời cả `tail`, `l1`, `l2` mỗi vòng?
- [ ] Trả về `dummy.next`, không phải `dummy`?
- [ ] Đã thử `5 + 5`, `999 + 1`, `0 + 0`?

---

## Khi nào nên nghĩ tới kỹ thuật này?

- "Số được biểu diễn bằng danh sách chữ số"
- "Không được chuyển sang số nguyên"
- "Trả về một Linked List mới"
- Có phép cộng / trừ "có nhớ" giữa hai dãy

> Mỗi khi phải tạo Linked List mới, hãy bắt đầu bằng `dummy = ListNode(); tail = dummy`.